# GARCH Model and its extensions

Historical volatility (HV) and EWMA underestimate the volatility spikes particularly during crisis periods as their reaction to new regime/shocks is slow.

GARCH can react faster because in this case the large squared returns directly shock volatility.

### GARCH(1,1)
$$ \sigma^2_t = \omega + \alpha r^2_{t-1} + \beta \sigma^2_{t-1}

In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\arbaz2\Desktop\Quant Finance\Volatility Forecasting\src")

from data_pipeline import make_dataset
from baselines import make_baseline_forecasts
from metrics import qlike, mse, score, score_by_regime, score_by_ticker

CRISIS_WINDOWS = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31"),
}

df = make_dataset(["SPY", "JPM"], start="2000-01-01", horizons=(1,5), crisis_windows=CRISIS_WINDOWS)
df["date"] = pd.to_datetime(df["date"])


In [2]:
# Add baseline forecasts so we can compare to GARCH
df_f = make_baseline_forecasts(df, hv_window=20, ewma_lam=0.94)
df_f.head(40)

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389446,4723500,-1.733122,3.003711,5.704489,14.393478,calm,NaN,NaN,5.291806,26.459029
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641876,5741700,0.342478,0.117291,1.449098,36.091977,calm,NaN,NaN,1.468518,7.342589
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861031,8405550,-2.388407,5.704489,0.390175,14.400223,calm,NaN,NaN,5.154520,25.772601
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545319,7503700,-1.203785,1.449098,0.999589,37.059621,calm,NaN,NaN,1.387444,6.937221
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998011,7271850,0.624640,0.390175,2.253553,14.666601,calm,NaN,NaN,5.187518,25.937591
5,2000-01-12,SPY,144.593750,144.593750,142.875000,143.062500,89.644562,6907700,-0.999795,0.999589,1.809643,36.244200,calm,NaN,NaN,1.391143,6.955717
6,2000-01-13,JPM,47.416668,48.333332,47.041668,47.541668,22.330732,6918900,1.501184,2.253553,12.462991,23.814919,calm,NaN,NaN,4.899678,24.498388
7,2000-01-13,SPY,144.468750,145.750000,143.281250,145.000000,90.858635,5158300,1.345230,1.809643,1.818701,6.194323,calm,NaN,NaN,1.367650,6.838251
8,2000-01-14,JPM,49.291668,50.500000,48.541668,49.250000,23.133154,9731850,3.530296,12.462991,15.756789,36.567997,calm,NaN,NaN,4.740910,23.704551
9,2000-01-14,SPY,146.531250,147.468750,145.968750,146.968750,92.092247,7437300,1.348592,1.818701,0.623850,6.700881,calm,NaN,NaN,1.394170,6.970849


hv1_var and hv5_var are NaN early on — that’s expected because rolling variance needs a full window (e.g., 20 days)

### Rolling GARCH(1,1) forecasts (no leakage)

The code below fits a GARCH model using only past data, refit on a schedule (monthly by default), and produces 1-day variance forecasts.

In [4]:
from arch import arch_model

def garch_rolling_forecast_1d_single(
    d: pd.DataFrame,
    dist: str = "t",
    mean: str = "zero",
    p: int = 1,
    q: int = 1,
    refit_every: int = 21,   # ~monthly
    min_train: int = 750,    # ~3 years of trading days
):
    """
    Rolling/refit GARCH(p,q) one-step-ahead variance forecast for a single ticker.

    Inputs
    ------
    d: DataFrame for ONE ticker containing columns ['date', 'ret'].
       Must be sorted by date ascending.

    Output
    ------
    pd.Series of garch1_var aligned to d.index at time t:
      forecast at index t uses data up to t-1 and predicts variance of return at t (or t+1 depending on alignment).
    Here we align it as: at row t, store 1-step ahead forecast produced after observing returns up to t-1.
    """
    d = d.sort_values("date").copy()
    r = d["ret"].astype(float).reset_index(drop=True)

    # We'll return a Series aligned to d's original index
    fcast = pd.Series(index=d.index, dtype=float)

    last_fit = None
    last_fit_end = None

    # We iterate over rows; at "t" we fit on returns up to t-1 and store forecast for t (one-step ahead).
    # This lines up nicely with your targets (rv1_var at time t corresponds to ret_{t+1}^2 in your pipeline),
    # because later you'll compare forecasts aligned at time t with target rv1_var at time t.
    #
    # In other words:
    # - At time t, your features use info up to t
    # - Your target is ret_{t+1}^2
    # So your forecast should also be "variance of ret_{t+1}" computed using data up to t.
    #
    # We'll implement that by fitting on r[:t+1] and forecasting next step at row t.
    #
    # To keep it simple and consistent, we’ll:
    # - Fit on r.iloc[:t+1] (includes ret_t)
    # - Store the 1-step ahead forecast aligned at time t
    #
    # This is the correct alignment for predicting rv1_var(t) = ret_{t+1}^2

    for t in range(len(d)):
        if t < min_train:
            continue

        # Refit periodically
        if (last_fit is None) or (last_fit_end is None) or ((t - last_fit_end) >= refit_every):
            train = r.iloc[: t + 1]  # includes ret_t; forecasting t+1
            am = arch_model(train, mean=mean, vol="GARCH", p=p, q=q, dist=dist, rescale=False)
            try:
                last_fit = am.fit(disp="off")
                last_fit_end = t
            except Exception:
                # If fit fails, keep previous fit (if any)
                pass

        if last_fit is None:
            continue

        try:
            # One-step ahead variance forecast (for next period)
            v = last_fit.forecast(horizon=1, reindex=False).variance.values[-1, 0]
            fcast.iloc[t] = float(v)
        except Exception:
            pass

    return fcast

In [5]:
def add_garch_forecasts(
    df_f: pd.DataFrame,
    dist: str = "t",
    mean: str = "zero",
    p: int = 1,
    q: int = 1,
    refit_every: int = 21,
    min_train: int = 750,
):
    """
    Add GARCH forecasts to a panel DataFrame without using groupby.apply (avoids pandas FutureWarning).

    Adds:
    - garch1_var: 1-day ahead variance forecast aligned at time t (predicts ret_{t+1} variance)
    - garch5_var: simple approximation 5 * garch1_var (later we can do true multi-step)

    Returns df sorted by ['date','ticker'].
    """
    out = df_f.copy()
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    garch_series_list = []

    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()
        s = garch_rolling_forecast_1d_single(
            d,
            dist=dist,
            mean=mean,
            p=p,
            q=q,
            refit_every=refit_every,
            min_train=min_train,
        )
        # s is aligned to d.index; store it in the master frame index
        garch_series_list.append(s.rename(tkr))

    # Combine forecasts back into one Series aligned to out.index
    # Each s has indices belonging to 'out' rows for that ticker; concat will align by index.
    garch_all = pd.concat(garch_series_list, axis=0).sort_index()
    out["garch1_var"] = garch_all

    # Simple 5-day approximation
    out["garch5_var"] = 5.0 * out["garch1_var"]

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [6]:
df_g = add_garch_forecasts(df_f, dist="t", mean="zero", refit_every=5, min_train=750)
df_g[["date","ticker","rv1_var","hv1_var","ewma1_var","garch1_var"]].dropna().head(10)

,date,ticker,rv1_var,hv1_var,ewma1_var,garch1_var
1500,2003-01-07,JPM,15.360199,13.218742,14.165035,13.930963
1501,2003-01-07,SPY,2.118830,2.254766,2.274126,2.149192
1502,2003-01-08,JPM,3.843596,12.681586,13.330226,13.930963
1503,2003-01-08,SPY,2.377242,1.943871,2.141360,2.149192
1504,2003-01-09,JPM,0.596347,12.279544,13.452025,13.930963
1505,2003-01-09,SPY,0.072356,1.972659,2.140008,2.149192
1506,2003-01-10,JPM,0.488525,12.344666,12.875519,13.930963
1507,2003-01-10,SPY,0.001038,1.973318,2.154242,2.149192
1508,2003-01-13,JPM,1.276724,12.335396,12.138769,13.930963
1509,2003-01-13,SPY,0.103666,1.973694,2.029329,2.149192


In [17]:
# How many GARCH forecasts did we produce?
print(df_g["garch1_var"].notna().mean())

# Should be NaN early (before min_train) and non-NaN later
for tkr in df_g["ticker"].unique():
    sub = df_g[df_g["ticker"] == tkr].sort_values("date")
    first_non_nan = sub["garch1_var"].first_valid_index()
    print(tkr, "first forecast row index:", first_non_nan)

0.8859662460088186
JPM first forecast row index: 1500
SPY first forecast row index: 1501


This means
* ~88.6% fo rows have GARCH forecasts
* First forecast appears around row ~1500 because min_train = 750 per ticker and we have two tickers
This is correct behaviour

In [73]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var"]
models_5d = ["hv5_var", "ewma5_var", "garch5_var"]

print(score(df_g, models_1d, "rv1_var", eval_start="2005-01-01"))
print("\n")
print(score(df_g, models_5d, "rv5_var", eval_start="2005-01-01"))

score_by_regime(df_g, models_1d, "rv1_var", eval_start="2005-01-01").head(30)

        model   target      n     QLIKE         MSE
2  garch1_var  rv1_var  10652  1.442664  226.253153
1   ewma1_var  rv1_var  10652  1.450791  226.174661
0     hv1_var  rv1_var  10652  1.490453  235.490336


        model   target      n     QLIKE          MSE
0     hv5_var  rv5_var  10652  2.890882  1456.070893
1   ewma5_var  rv5_var  10652  2.911794  1330.494447
2  garch5_var  rv5_var  10652  2.945043  1369.998407


,regime,model,target,n,QLIKE,MSE
0,COVID_2020,hv1_var,rv1_var,144,4.259606,1823.981555
1,COVID_2020,ewma1_var,rv1_var,144,4.591991,1750.485173
2,COVID_2020,garch1_var,rv1_var,144,4.633548,1706.115400
5,GFC_2007_2009,garch1_var,rv1_var,1008,3.031297,1941.907183
4,GFC_2007_2009,ewma1_var,rv1_var,1008,3.034460,1938.604012
3,GFC_2007_2009,hv1_var,rv1_var,1008,3.092550,2022.670991
8,calm,garch1_var,rv1_var,9500,1.225735,21.781634
7,calm,ewma1_var,rv1_var,9500,1.235142,21.371556
6,calm,hv1_var,rv1_var,9500,1.278488,21.782880


In [25]:
def add_crisis_shading(fig, start, end, name, crisis_windows, opacity=0.2):
    if start is None and end is None:

        for name, (start, end) in crisis_windows.items():
            fig.add_vrect(
                x0=pd.to_datetime(start),
                x1=pd.to_datetime(end),
                fillcolor="gray",
                opacity=opacity,
                line_width=0,
                annotation_text=name,
                annotation_position="top left"
            )
    else:
        fig.add_vrect(
                x0=pd.to_datetime(start),
                x1=pd.to_datetime(end),
                fillcolor="gray",
                opacity=opacity,
                line_width=0,
                annotation_text=name,
                annotation_position="top left"
            )

    return fig

import plotly.graph_objects as go
def plot_vol_compare(
    df,
    ticker,
    crisis_windows=CRISIS_WINDOWS,
    target_var="rv1_var",
    forecast_vars=("hv1_var", "ewma1_var", "garch1_var_scaled"),
    start=None,
    end=None,
    name=None,
    title=None
):
    """
    Plot sqrt(variance) so it's more interpretable.
    target_var and forecast_vars should all be in percent^2 units.
    """
    d = df[df["ticker"] == ticker].sort_values("date").copy()
    if start is not None:
        d = d[d["date"] >= pd.to_datetime(start)]
    if end is not None:
        d = d[d["date"] <= pd.to_datetime(end)]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=d["date"], y=np.sqrt(d[target_var]),
        mode="lines", name="Realized vol"
    ))

    for col in forecast_vars:
        fig.add_trace(go.Scatter(
            x=d["date"], y=np.sqrt(d[col]),
            mode="lines", name=col
        ))

    fig.update_layout(
        title=title or f"{ticker}: Forecast vs Realized Vol",
        xaxis_title="Date",
        yaxis_title="Vol (same units as returns)"
    )
    add_crisis_shading(fig, start, end, name, crisis_windows)
    fig.show()

In [26]:
plot_vol_compare(df_g, "SPY", target_var="rv1_var",
                 forecast_vars=("hv1_var","ewma1_var","garch1_var"))

#### A naive GARCH implementation produced flat forecasts between refits (a deployment bug) leading to GARCH performing worse that EWMA paricularly during COVID (sudden market shock) as between refits, the forecast was not being updated with new returns thus remaining constant until the next refit. Increasing the refit frequency from every 21 days to every 5 days improved accuracy but was computationally expensive. This can be fixed by refitting parameters periodically while updating the conditional variance recursion daily, eliminating flat forecasts and yielding stable performance across regimes.

In [12]:
# Helper function to fit parameters at refit points

def fit_garch_params(ret_series, mean="zero", dist="t", p=1, q=1):
    """
    Fit GARCH(p,q) on a return series (in percent units to match your current pipeline).
    Returns (omega, alpha, beta) for GARCH(1,1). For p=q=1 only.
    """
    am = arch_model(ret_series, mean=mean, vol="GARCH", p=p, q=q, dist=dist, rescale=False)
    res = am.fit(disp="off")
    params = res.params

    # For GARCH(1,1): omega, alpha[1], beta[1]
    omega = float(params["omega"])
    alpha = float(params[[k for k in params.index if "alpha[1]" in k][0]])
    beta  = float(params[[k for k in params.index if "beta[1]"  in k][0]])

    return omega, alpha, beta

In [13]:
# Code to implement rolling refit and daily recursion for a single ticker
def garch_refit_and_recurse_1d(
    d: pd.DataFrame,
    refit_every=21,
    min_train=750,
    mean="zero",
    dist="t",
):
    """
    Correct GARCH(1,1) 1-step-ahead variance forecast:
    - Refit parameters every refit_every days on expanding window
    - Update conditional variance DAILY via recursion between refits
    - Output aligned at time t as forecast of Var(ret_{t+1}) in percent^2

    This eliminates the flat-stretch problem.
    """
    d = d.sort_values("date").copy()
    ret = d["ret"].astype(float).reset_index(drop=True)      # percent returns
    r2  = (ret ** 2)

    fcast = pd.Series(index=d.index, dtype=float)

    omega = alpha = beta = None
    sigma2_t = None

    # initialize sigma2_t with sample variance once we have min_train
    for t in range(len(d)):
        if t < min_train:
            continue

        # refit parameters on schedule (or first time)
        if (omega is None) or ((t - min_train) % refit_every == 0):
            train = ret.iloc[: t + 1]
            try:
                omega, alpha, beta = fit_garch_params(train, mean=mean, dist=dist)
            except Exception:
                # if fit fails, keep previous params (if any)
                pass

            # initialize sigma2_t at refit time using recent variance
            if sigma2_t is None:
                sigma2_t = float(np.var(train.values, ddof=1))

        if omega is None:
            continue

        # Forecast Var(ret_{t+1}) using recursion based on information up to t
        # sigma2_{t+1} = omega + alpha * r_t^2 + beta * sigma2_t
        # Store forecast aligned at time t:
        sigma2_next = omega + alpha * float(r2.iloc[t]) + beta * float(sigma2_t)
        fcast.iloc[t] = sigma2_next

        # Advance state for next step
        sigma2_t = sigma2_next

    return fcast

In [14]:
# Wrapper function
def add_garch_refit_recurse(df_f, refit_every=21, min_train=750, mean="zero", dist="t"):
    out = df_f.copy().sort_values(["ticker","date"]).reset_index(drop=True)

    pieces = []
    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()
        s = garch_refit_and_recurse_1d(
            d,
            refit_every=refit_every,
            min_train=min_train,
            mean=mean,
            dist=dist,
        )
        pieces.append(s.rename(tkr))

    garch_all = pd.concat(pieces, axis=0).sort_index()
    out["garch1_var"] = garch_all
    out["garch5_var"] = 5.0 * out["garch1_var"]

    return out.sort_values(["date","ticker"]).reset_index(drop=True)

In [15]:
df_garch = add_garch_refit_recurse(df_f, refit_every=5, min_train=750, mean="zero", dist="t")

In [49]:


models_1d = ["hv1_var", "ewma1_var", "garch1_var"]
print(score(df_garch, models_1d, "rv1_var", eval_start="2005-01-01"))
print("\n")

models_5d = ["hv5_var", "ewma5_var", "garch5_var"]
print(score(df_garch, models_5d, "rv5_var", eval_start="2005-01-01"))
print("\n")
print(score_by_regime(df_garch, models_1d, "rv1_var", eval_start="2005-01-01").head(30))

        model   target      n     QLIKE         MSE
2  garch1_var  rv1_var  10652  1.393296  220.019929
1   ewma1_var  rv1_var  10652  1.450791  226.174661
0     hv1_var  rv1_var  10652  1.490453  235.490336


        model   target      n     QLIKE          MSE
2  garch5_var  rv5_var  10652  2.844595  1058.148220
0     hv5_var  rv5_var  10652  2.890882  1456.070893
1   ewma5_var  rv5_var  10652  2.911794  1330.494447


          regime       model   target     n     QLIKE          MSE
2     COVID_2020  garch1_var  rv1_var   144  3.968346  1576.128409
0     COVID_2020     hv1_var  rv1_var   144  4.259606  1823.981555
1     COVID_2020   ewma1_var  rv1_var   144  4.591991  1750.485173
5  GFC_2007_2009  garch1_var  rv1_var  1008  3.004557  1900.091876
4  GFC_2007_2009   ewma1_var  rv1_var  1008  3.034460  1938.604012
3  GFC_2007_2009     hv1_var  rv1_var  1008  3.092550  2022.670991
8           calm  garch1_var  rv1_var  9500  1.183301    21.199704
7           calm   ewma1_var  rv1_var  9

In [50]:
plot_vol_compare(df_garch, "SPY", target_var="rv1_var",
                 forecast_vars=("hv1_var","ewma1_var","garch1_var"))

In [24]:
plot_vol_compare(
    df_garch, "SPY",
    target_var="rv1_var",
    forecast_vars=("hv1_var","ewma1_var","garch1_var"),
    start="2020-02-15", end="2020-05-31", name = "COVID",
    title="SPY: HV vs EWMA vs GARCH (refit+recurse) during COVID"
)

#### Now with refiting and using recursion, the GARCH clearly beats EWMA and HV for both time horizons (1-day and 5-day) and for all regimes (calm, GFC and COVID).be 
#### The forecasted volatility spikes are still smaller than the realized ones. This is expected as the target rv1_vat(t) = ret_{t+1}^2 is one-day realized variance which is dominated by jumps. A 1-day return can suddenly be $\pm$ 8% with essentially no warning. Any conditional variance model (EWMA, GARCH, HAR) is forecasting the conditional expectation, not the jump itself.

So:
* Realized variance spikes = "what happened"
* Forecasted variance spikes = " waht the model expected could happen given the information up to t"

The forecast spikes typically:
* rise after the first big shock day
* not match the peak of realized spike
* the decay gradually.

This is normal even in production


#### GARCH spikes look similar to EWMA
For daily data, GARCH(1,1) and EWMA are often close. In fact, EWMA is basically a special case of an IGARCH-like recursion (conceptually), and empirically they can track each other closely. So there isn't usually a visually dramatic difference day-to-day. The improvement shows up in the loss function across long samples and regimes just as noticed above.

Where GARCH helps is typically:
* slightly better persistence structure ( $ \alpha + \beta$ close to 1 but not forced)
* a learned $\omega$ term (mean reversion level)
* sometimes better decay after shocks
* sometimes better medium-horizon forecasts (e.g., the 5-day results above)

#### refit_every = 5 and 21 give similar results
This is because the "daily-recursion update" is doing the most work. Refitting parameters more often doesn't change much -- which is exactly what we want i.e.,
* stable parameters
* correct state update daily
This is a good result: it shows that the model is not overly sensitive to refit schedule.



In [53]:

def plot_error_vol(df, ticker, target="rv1_var",
                   models=("ewma1_var","garch1_var","hv1_var"),
                   start=None, end=None, title=None):
    d = df[df["ticker"] == ticker].sort_values("date").copy()
    if start: d = d[d["date"] >= pd.to_datetime(start)]
    if end:   d = d[d["date"] <= pd.to_datetime(end)]

    realized = np.sqrt(d[target].values)

    fig = go.Figure()
    for m in models:
        err = np.sqrt(d[m].values) - realized
        fig.add_trace(go.Scatter(x=d["date"], y=err, mode="lines", name=f"{m} error"))

    fig.add_hline(y=0, line_width=1)
    fig.update_layout(
        title=title or f"{ticker}: Vol Forecast Error (sqrt(var) space)",
        xaxis_title="Date",
        yaxis_title="Forecast vol − Realized vol"
    )
    fig.show()

# COVID onset zoom
plot_error_vol(df_garch, "SPY", start="2020-02-15", end="2020-05-31",
               title="SPY: Vol Error During COVID (EWMA vs GARCH vs HV)")

# GFC onset zoom
plot_error_vol(df_garch, "SPY", start="2007-07-01", end="2009-06-30",
               title="SPY: Vol Error During GFC (EWMA vs GARCH vs HV)")

In [28]:
import plotly.express as px

def calibration_plot(df, ticker, model, target="rv1_var", eval_start="2005-01-01", q=10):
    d = df[(df["ticker"] == ticker) & (df["date"] >= pd.to_datetime(eval_start))].copy()
    d = d.dropna(subset=[model, target])

    # Quantile bins of the forecast (each bin has ~equal number of points)
    d["bin"] = pd.qcut(d[model], q=q, labels=False, duplicates="drop")

    # Compare average forecast vs average realized within each bin
    cal = d.groupby("bin").agg(
        forecast_mean=(model, "mean"),
        realized_mean=(target, "mean"),
        n=(target, "size")
    ).reset_index()

    fig = px.scatter(
        cal, x="forecast_mean", y="realized_mean", size="n",
        title=f"{ticker}: Calibration of {model} (binned, q={q})",
        labels={"forecast_mean":"Avg forecast variance", "realized_mean":"Avg realized variance"}
    )
    # 45-degree line (perfect calibration)
    lo = min(cal["forecast_mean"].min(), cal["realized_mean"].min())
    hi = max(cal["forecast_mean"].max(), cal["realized_mean"].max())
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="Perfect calibration"))
    fig.show()

calibration_plot(df_g, "SPY", "ewma1_var")
calibration_plot(df_g, "SPY", "garch1_var")
calibration_plot(df_g, "SPY", "hv1_var")

In [29]:

def garch_param_trace(df, ticker, refit_every=21, min_train=750,
                      mean="zero", dist="t", eval_start="2005-01-01"):
    d = df[df["ticker"] == ticker].sort_values("date").copy()
    d = d[d["date"] >= pd.to_datetime("2000-01-01")]  # ensure full history
    ret = d["ret"].astype(float).reset_index(drop=True)

    rows = []
    for t in range(min_train, len(d), refit_every):
        train = ret.iloc[: t + 1]
        try:
            am = arch_model(train, mean=mean, vol="GARCH", p=1, q=1, dist=dist, rescale=False)
            res = am.fit(disp="off")
            p = res.params
            omega = float(p["omega"])
            alpha = float(p[[k for k in p.index if "alpha[1]" in k][0]])
            beta  = float(p[[k for k in p.index if "beta[1]"  in k][0]])
            rows.append({"date": d["date"].iloc[t], "omega": omega, "alpha": alpha, "beta": beta, "a_plus_b": alpha+beta})
        except Exception:
            continue

    par = pd.DataFrame(rows)
    par = par[par["date"] >= pd.to_datetime(eval_start)]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=par["date"], y=par["alpha"], mode="lines", name="alpha"))
    fig.add_trace(go.Scatter(x=par["date"], y=par["beta"], mode="lines", name="beta"))
    fig.add_trace(go.Scatter(x=par["date"], y=par["a_plus_b"], mode="lines", name="alpha+beta"))
    fig.update_layout(
        title=f"{ticker}: GARCH(1,1) parameter trace (refit_every={refit_every})",
        xaxis_title="Date",
        yaxis_title="Value"
    )
    fig.show()

garch_param_trace(df_g, "SPY", refit_every=21, min_train=750, dist="t")
garch_param_trace(df_g, "JPM", refit_every=21, min_train=750, dist="t")

# GJR GARCH

Since GARCH(1,1) models volatility as
$$ \sigma^2_t = \omega + \alpha r^2_{t-1} + \beta \sigma^2_{t-1} $$

where $\omega$ represents the long-run variance level, $\alpha$ reaction to shocks and $\beta$ persistence. Therefore, volatility increases after large shocks and is independent of the sign of returns. But in financial markets there is a well-known asymmetry.

**Levarage Effect: Negative returns increase volatility more than positive returns of the same magnitude.**

GJR-GARCH adds an asymmetric term to model the leverage effect as:
$$ \sigma^2_t = \omega + \alpha r^2_{t-1} + \beta \sigma^2_{t-1} + \gamma r^2_{t-1} I (r_{t-1} < 0) $$
where
* $I (r_{t-1} < 0)$ is the indicator for negative return
* $\gamma$ measures extra volatility impact from negative shocks

GJR-GARCH slightly beats GARCH in crisis periods while giving similar performance in calm periods.

In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd

def gjr_garch_rolling_forecast_1d_single(
    d,
    refit_every=21,
    min_train=750,
    mean="zero",
    dist="t"
):
    
    d = d.sort_values("date").copy() # ensure chronological order
    ret = d["ret"].astype(float).values #extract returns
    r2 = ret ** 2 #sqaured returns
    
    fcast = np.full(len(d), np.nan)
    
    sigma2_t = None
    t = min_train
    
    while t < len(d):
        
        train = ret[:t+1]
        
        try:
            am = arch_model(
                train,
                mean=mean,
                vol="GARCH",
                p=1,
                o=1,        # <- THIS activates GJR asymmetry
                q=1,
                dist=dist,
                rescale=False
            )
            
            res = am.fit(disp="off")
            
            p = res.params
            
            omega = float(p["omega"])
            alpha = float(p[[k for k in p.index if "alpha[1]" in k][0]])
            gamma = float(p[[k for k in p.index if "gamma[1]" in k][0]])
            beta  = float(p[[k for k in p.index if "beta[1]"  in k][0]])
        
        except Exception:
            t += refit_every
            continue
        
        if sigma2_t is None:
            sigma2_t = np.var(train, ddof=1)
        
        t_end = min(t + refit_every, len(d))
        
        for i in range(t, t_end):
            
            indicator = 1.0 if ret[i] < 0 else 0.0
            
            sigma2_next = (
                omega
                + alpha * r2[i]
                + gamma * r2[i] * indicator
                + beta * sigma2_t
            )
            
            fcast[i] = sigma2_next
            sigma2_t = sigma2_next
        
        t = t_end
    
    return pd.Series(fcast, index=d.index)

In [37]:
def add_gjr_garch_forecast(
    df_f,
    refit_every=21,
    min_train=750,
    mean="zero",
    dist="t"
):
    
    out = df_f.copy().sort_values(["ticker","date"]).reset_index(drop=True)
    
    forecasts = []
    
    for tkr in out["ticker"].unique():
        
        d = out[out["ticker"] == tkr].copy()
        
        s = gjr_garch_rolling_forecast_1d_single(
            d,
            refit_every=refit_every,
            min_train=min_train,
            mean=mean,
            dist=dist
        )
        
        forecasts.append(s.rename(tkr))
    
    gjr_all = pd.concat(forecasts, axis=0).sort_index()
    
    out["gjr1_var"] = gjr_all
    out["gjr5_var"] = 5.0 * out["gjr1_var"]
    
    return out.sort_values(["date","ticker"]).reset_index(drop=True)

In [54]:
df_gjr_garch = add_gjr_garch_forecast(
    df_garch,
    refit_every=21,
    min_train=750,
    dist="t"
)

In [59]:
models_1d = ["hv1_var","ewma1_var","garch1_var","gjr1_var"]
print(score(df_gjr_garch, models_1d, "rv1_var", eval_start="2005-01-01"))
print("\n")

models_5d = ["hv5_var", "ewma5_var", "garch5_var","gjr5_var"]
print(score(df_gjr_garch, models_5d, "rv5_var", eval_start="2005-01-01"))
print("\n")
print(score_by_regime(df_gjr_garch, models_1d, "rv1_var", eval_start="2005-01-01").head(30))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10652  1.359027  213.055694
2  garch1_var  rv1_var  10652  1.393296  220.019929
1   ewma1_var  rv1_var  10652  1.450791  226.174661
0     hv1_var  rv1_var  10652  1.490453  235.490336


        model   target      n     QLIKE          MSE
3    gjr5_var  rv5_var  10652  2.831690   999.880002
2  garch5_var  rv5_var  10652  2.844595  1058.148220
0     hv5_var  rv5_var  10652  2.890882  1456.070893
1   ewma5_var  rv5_var  10652  2.911794  1330.494447


           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724728  1546.464409
2      COVID_2020  garch1_var  rv1_var   144  3.968346  1576.128409
0      COVID_2020     hv1_var  rv1_var   144  4.259606  1823.981555
1      COVID_2020   ewma1_var  rv1_var   144  4.591991  1750.485173
7   GFC_2007_2009    gjr1_var  rv1_var  1008  2.948959  1834.596462
6   GFC_2007_2009  garch1_var  rv1_var  1008  3.004557  1900.09

As we can see, GJR-GARCH gives even better results particularly in crisis regimes

In [61]:
plot_vol_compare(df_gjr_garch, "SPY", target_var="rv1_var",
                 forecast_vars=("hv1_var","ewma1_var","garch1_var", "gjr1_var"))

In [68]:
models=("ewma1_var","garch1_var","hv1_var", "gjr1_var")
# COVID onset zoom
plot_error_vol(df_gjr_garch, "SPY", models=("ewma1_var","garch1_var","hv1_var", "gjr1_var"), start="2020-02-15", end="2020-05-31",
               title="SPY: Vol Error During COVID (EWMA vs GARCH vs HV)")

# GFC onset zoom
plot_error_vol(df_gjr_garch, "SPY", models = ("ewma1_var","garch1_var","hv1_var", "gjr1_var"), start="2007-07-01", end="2009-06-30",
               title="SPY: Vol Error During GFC (EWMA vs GARCH vs HV)")